<a href="https://colab.research.google.com/github/AnanyaAsthana/Machine-Learning/blob/main/ROC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Load Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# Convert to binary classification
# Setosa (class 0) -> Positive (1)
# Others -> Negative (0)
y_binary = (y == 0).astype(int)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.3, random_state=42, stratify=y_binary
)

# Train Naive Bayes classifier
model = GaussianNB()
model.fit(X_train, y_train)

# Get probability scores for positive class
y_scores = model.predict_proba(X_test)[:, 1]

# Manual ROC and AUC computation
def compute_roc_auc_manual(y_true, y_score):
    thresholds = np.sort(np.unique(y_score))[::-1]
    P = np.sum(y_true == 1)
    N = np.sum(y_true == 0)

    roc_points = []

    for t in thresholds:
        y_pred = (y_score >= t).astype(int)

        TP = np.sum((y_pred == 1) & (y_true == 1))
        FP = np.sum((y_pred == 1) & (y_true == 0))

        TPR = TP / P
        FPR = FP / N

        roc_points.append((FPR, TPR, t))

    # Sort by FPR
    roc_points = sorted(roc_points, key=lambda x: x[0])

    fprs = [p[0] for p in roc_points]
    tprs = [p[1] for p in roc_points]

    auc = np.trapz(tprs, fprs)

    return auc, roc_points

# Manual AUC
manual_auc, roc_points = compute_roc_auc_manual(y_test, y_scores)

# Built-in AUC
sklearn_auc = roc_auc_score(y_test, y_scores)

# Find optimal threshold (nearest to (0,1))
best_distance = float("inf")
best_point = None

for fpr, tpr, thresh in roc_points:
    distance = np.sqrt((fpr)**2 + (1 - tpr)**2)
    if distance < best_distance:
        best_distance = distance
        best_point = (fpr, tpr, thresh)

# Print results
print("Manual AUC :", manual_auc)
print("Sklearn AUC:", sklearn_auc)

print("\nNearest ROC point to (0,1):")
print("FPR :", best_point[0])
print("TPR :", best_point[1])
print("Optimal Threshold :", best_point[2])


Manual AUC : 1.0
Sklearn AUC: 1.0

Nearest ROC point to (0,1):
FPR : 0.0
TPR : 1.0
Optimal Threshold : 0.9999991755363627


/tmp/ipython-input-1144999551.py:54: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc = np.trapz(tprs, fprs)
